# Titano V8 - Aggressive Sniper (Anti-Zero-Action-Collapse)
Esegui queste celle in sequenza per addestrare un bot che non ha paura di operare.

In [ ]:
!pip install stable-baselines3[extra] yfinance gymnasium pandas numpy

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv
import time
import os


In [ ]:
ASSETS = [
    "BTC-USD", "ETH-USD", "SOL-USD", "BNB-USD", "XRP-USD", "ADA-USD", "AVAX-USD", "DOGE-USD", "DOT-USD", "LTC-USD",
    "^GSPC", "^IXIC", "^DJI", "^RUT", "^VIX", "^FTSE", "^GDAXI", "^FCHI", "^N225", "^HSI",
    "GC=F", "CL=F", "NG=F", "SI=F", "HG=F", "ZC=F", "ZO=F", "KE=F", "ZR=F", "GF=F",
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "TSLA", "AVGO", "CSCO", "CRM",
    "JPM", "BAC", "WFC", "C", "GS", "MS", "AXP", "V", "MA", "PYPL",
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "TMO", "DHR", "ABT", "BMY", "AMGN",
    "PG", "KO", "PEP", "WMT", "COST", "MCD", "NKE", "SBUX", "HD", "LOW",
    "XOM", "CVX", "COP", "SLB", "EOG", "PXD", "MPC", "PSX", "VLO", "OXY",
    "BA", "CAT", "GE", "MMM", "HON", "UNP", "UPS", "RTX", "LMT", "DE",
    "SPY", "QQQ", "DIA", "IWM", "EEM", "VTI", "ARKK", "GLD", "SLV", "USO"
]

print("Scaricamento dati a 1 minuto per 100 asset (ultimi 7 giorni)...")
data_frames = {}
for symbol in ASSETS:
    try:
        df = yf.download(symbol, period="7d", interval="1m", progress=False)
        if not df.empty and len(df) > 1000:
            df['Returns'] = df['Close'].pct_change()
            df['Volatility'] = df['Close'].rolling(window=10).std()
            df.dropna(inplace=True)
            data_frames[symbol] = df
    except Exception as e:
        pass
print(f"Scaricati {len(data_frames)} asset con successo.")

In [ ]:
class AdvancedPortfolioEnv(gym.Env):
    def __init__(self, data_frames):
        super(AdvancedPortfolioEnv, self).__init__()
        self.data_frames = data_frames
        self.asset_names = list(data_frames.keys())
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(120,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)
        self.initial_capital = 100000.0
        self.capital = self.initial_capital
        self.total_shorts = 0
        self.total_longs = 0
        self.winning_trades = 0
        self.losing_trades = 0
        self.bankruptcies = 0
        self.position_steps = 0
        self.reset()
        
    def _get_obs(self):
        df = self.data_frames[self.current_asset]
        idx = self.current_step
        if idx >= len(df):
            return np.zeros(120, dtype=np.float32)
        obs = np.zeros(120, dtype=np.float32)
        start_idx = max(0, idx - 60)
        recent_data = df.iloc[start_idx:idx]
        returns = recent_data['Returns'].values
        volatility = recent_data['Volatility'].values
        obs[0:len(returns)] = returns
        obs[60:60+len(volatility)] = volatility
        return obs

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.capital = self.initial_capital
        self.current_asset = np.random.choice(self.asset_names)
        max_start = max(1, len(self.data_frames[self.current_asset]) - 1440)
        self.current_step = np.random.randint(60, max_start)
        self.end_step = min(len(self.data_frames[self.current_asset]) - 1, self.current_step + 1440)
        self.position = 0
        self.entry_price = 0.0
        self.step_counter = 0
        self.position_steps = 0
        return self._get_obs(), {}

    def step(self, action):
        df = self.data_frames[self.current_asset]
        current_price = float(df['Close'].iloc[self.current_step])
        reward = 0.0
        done = False
        
        # AZIONE 2: BUY/LONG (Inversione o Nuova posizione al rialzo)
        if action == 2:
            self.total_longs += 1
            if self.position == -1:
                # Inversione: Chiude lo Short e si prepara per il Long
                pnl_pct = (self.entry_price - current_price) / self.entry_price
                reward += self._calculate_reward(pnl_pct)
                self.position_steps = 0
            if self.position != 1:
                self.position = 1
                self.entry_price = current_price
                self.position_steps = 0
                
        # AZIONE 0: SELL/SHORT (Inversione o Nuova posizione al ribasso)
        elif action == 0:
            self.total_shorts += 1
            if self.position == 1:
                # Inversione: Chiude il Long e si prepara per lo Short
                pnl_pct = (current_price - self.entry_price) / self.entry_price
                reward += self._calculate_reward(pnl_pct)
                self.position_steps = 0
            if self.position != -1:
                self.position = -1
                self.entry_price = current_price
                self.position_steps = 0
                
        # AZIONE 1: FLAT/ATTESA (Chiude tutto e resta a guardare)
        elif action == 1:
            if self.position == 1:
                # Chiude Long
                pnl_pct = (current_price - self.entry_price) / self.entry_price
                reward += self._calculate_reward(pnl_pct)
                self.position = 0
                self.position_steps = 0
            elif self.position == -1:
                # Chiude Short
                pnl_pct = (self.entry_price - current_price) / self.entry_price
                reward += self._calculate_reward(pnl_pct)
                self.position = 0
                self.position_steps = 0

        # GESTIONE DELLO STATO CORRENTE E PENALITÀ FLOATING
        if self.position == 1:
            self.position_steps += 1
            # Penalità continua se il trade LONG sta perdendo
            pnl_pct = (current_price - self.entry_price) / self.entry_price
            if pnl_pct < 0:
                reward += pnl_pct * 200
                
            # REGOLA DELLE 4 ORE (240 minuti/step) DEL SUPERVISORE (Solo multa, nessuna chiusura)
            if self.position_steps == 240:
                # Applica la multa dell'Esattore in base al PnL come nel vero Supervisore
                pnl_pct = (current_price - self.entry_price) / self.entry_price
                if -0.01 <= pnl_pct <= 0.01:
                    reward -= 10.0 # Stagnazione
                elif -0.02 <= pnl_pct < -0.01:
                    reward -= 16.0 # Sanguinamento Lento
                elif 0.01 < pnl_pct <= 0.02:
                    reward -= 4.0  # Profitto Lento
                elif pnl_pct > 0.02:
                    reward -= 10.0 # Avidità
                elif pnl_pct < -0.02:
                    reward -= 20.0 # Testardaggine
                
        elif self.position == -1:
            self.position_steps += 1
            # Penalità continua se il trade SHORT sta perdendo
            pnl_pct = (self.entry_price - current_price) / self.entry_price
            if pnl_pct < 0:
                reward += pnl_pct * 200
                
            # REGOLA DELLE 4 ORE (240 minuti/step) DEL SUPERVISORE (Solo multa, nessuna chiusura)
            if self.position_steps == 240:
                # Applica la multa dell'Esattore in base al PnL come nel vero Supervisore
                pnl_pct = (self.entry_price - current_price) / self.entry_price
                if -0.01 <= pnl_pct <= 0.01:
                    reward -= 10.0 # Stagnazione
                elif -0.02 <= pnl_pct < -0.01:
                    reward -= 16.0 # Sanguinamento Lento
                elif 0.01 < pnl_pct <= 0.02:
                    reward -= 4.0  # Profitto Lento
                elif pnl_pct > 0.02:
                    reward -= 10.0 # Avidità
                elif pnl_pct < -0.02:
                    reward -= 20.0 # Testardaggine
                
        elif self.position == 0:
            # PENALITÀ PER INATTIVITÀ (Il costo del tempo perso)
            reward -= 2.0

        self.current_step += 1
        self.step_counter += 1
        
        if self.capital <= 10000:
            # Bancarotta ridotta per non traumatizzare la rete
            reward -= 50000.0
            self.bankruptcies += 1
            done = True
            
        if self.step_counter >= 1440 or self.current_step >= self.end_step:
            if self.position == 1:
                # Chiusura fine giornata
                pnl_pct = (current_price - self.entry_price) / self.entry_price
                reward += self._calculate_reward(pnl_pct)
                reward -= 500.0
            done = True
            
        return self._get_obs(), reward, done, False, {}

    def _calculate_reward(self, pnl_pct):
        trade_pnl_usd = (self.capital * 0.10) * pnl_pct
        self.capital += trade_pnl_usd
        if pnl_pct > 0:
            self.winning_trades += 1
            return pnl_pct * 1000  # Premia di più le vittorie!
        else:
            self.losing_trades += 1
            return pnl_pct * 200


In [ ]:
class LiveMetricsCallback(BaseCallback):
    def __init__(self, env, check_freq=3000, verbose=1):
        super(LiveMetricsCallback, self).__init__(verbose)
        self.env = env
        self.last_time = time.time()
        
    def _on_step(self) -> bool:
        current_time = time.time()
        if current_time - self.last_time >= 180:
            self.last_time = current_time
            e = self.env.envs[0].unwrapped
            total_trades = e.winning_trades + e.losing_trades
            win_rate = (e.winning_trades / total_trades * 100) if total_trades > 0 else 0
            print("\n" + "="*60)
            print("📊 METRICHE FINANZIARIE IN TEMPO REALE")
            print("="*60)
            print(f"Capitale Residuo: ${e.capital:,.2f}")
            print(f"Acquisti al Rialzo (LONG):  {e.total_longs}")
            print(f"Acquisti al Ribasso (SHORT): {e.total_shorts}")
            print(f"Win Rate: {win_rate:.1f}%")
            print(f"Bancarotte: {e.bankruptcies}")
            print("\n🧠 METRICHE NEURALI (Ultimo Batch)")
            print("="*60)
            if self.logger:
                name_to_value = self.logger.name_to_value
                for key, val in name_to_value.items():
                    if 'train/' in key:
                        print(f"{key.replace('train/', '')}: {val:.5f}")
            print("="*60 + "\n")
        return True


In [ ]:
env = DummyVecEnv([lambda: AdvancedPortfolioEnv(data_frames)])
print("ATTENZIONE: Nessun modello precedente richiesto, partiamo da zero per V8!")
model = PPO("MlpPolicy", env, verbose=0)
callback = LiveMetricsCallback(env)
print("Inizio Addestramento V8 Aggressive da 50 Milioni di Step...")
model.learn(total_timesteps=50_000_000, callback=callback, reset_num_timesteps=False)
print("Addestramento Completato!")
model.save("Titano_V8_Aggressive.zip")
print("Nuovo modello V8 salvato. Scaricalo da Colab!")
